In [1]:
#Library
import time
import urllib.parse
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

In [2]:
#Konfigurasi Chrome
def create_driver(use_path=False):
    chrome_options = Options()
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-notifications")
    chrome_options.add_argument("--disable-infobars")
    chrome_options.add_experimental_option("excludeSwitches", ['enable-automation'])
    chrome_options.add_argument("--window-size=1300,800")
    chrome_options.add_argument("--window-position=200,100")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                                "AppleWebKit/537.36 (KHTML, like Gecko) "
                                "Chrome/120.0.0.0 Safari/537.36")
    if use_path:
        path = r"C:\Windows\chromedriver.exe"  #path manual
        service = Service(path)
    else:
        service = Service(ChromeDriverManager().install())  #otomatis download
    driver = webdriver.Chrome(service=service, options=chrome_options)
    return driver

In [3]:
#Scraping
#Membuka tokopedia
def scrape_tokopedia(keyword, max_scroll=20, wait_time=3, use_path=False):
    keyword_encoded = urllib.parse.quote(keyword)
    main_link = f"https://www.tokopedia.com/search?st=&q={keyword_encoded}"
    driver = create_driver(use_path=use_path)
    driver.get(main_link)
    time.sleep(wait_time)
    last_count = 0
    for i in range(max_scroll):
        
        #Scroll otomatis
        driver.execute_script("window.scrollBy(0, 1000);")
        time.sleep(wait_time)

        #Deteksi produk
        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")
        produk_containers = soup.find_all("div", class_="gG1uA844gIiB2+C3QWiaKA==")
        
        print(f"Scroll {i+1}: {len(produk_containers)} produk ditemukan")
        if len(produk_containers) == last_count:
            print("Tidak ada produk baru, berhenti scrolling.")
            break
        last_count = len(produk_containers)

    #Mendeteksi box produk
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")
    produk_containers = soup.find_all("div", class_="gG1uA844gIiB2+C3QWiaKA==")
    
    #Mendeteksi data berdasarkan HTML
    data = []
    for produk in produk_containers:
        nama_produk = produk.find("span", class_="+tnoqZhn89+NHUA43BpiJg==")
        harga = produk.find("div", class_=[
            "urMOIDHH7I0Iy1Dv2oFaNw== HJhoi0tEIlowsgSNDNWVXg==",
            "urMOIDHH7I0Iy1Dv2oFaNw=="
        ])
        promo = produk.find("span", class_="_7UCYdN8MrOTwg0MKcGu8zg==")
        rating = produk.find("span", class_="_2NfJxPu4JC-55aCJ8bEsyw==")
        terjual = produk.find("span", class_="u6SfjDD2WiBlNW7zHmzRhQ==")
        penjual = produk.find("span", class_="si3CNdiG8AR0EaXvf6bFbQ== gxi+fsEljOjqhjSKqjE+sw== flip")
        lokasi = produk.find("span", class_="gxi+fsEljOjqhjSKqjE+sw== flip")
        #Penyimpanan dalam dataframe
        data.append({
            "Produk": nama_produk.get_text(strip=True) if nama_produk else None,
            "Harga": harga.get_text(strip=True) if harga else None,
            "Promo": promo.get_text(strip=True) if promo else None,
            "Rating": rating.get_text(strip=True) if rating else None,
            "Terjual": terjual.get_text(strip=True) if terjual else None,
            "Penjual": penjual.get_text(strip=True) if penjual else None,
            "Lokasi": lokasi.get_text(strip=True) if lokasi else None,
        })

    driver.quit()
    return pd.DataFrame(data)

In [4]:
# Live scraping is opt-in: Tokopedia's markup changes often and scraping a live
# commercial site is not a reproducible portfolio step. Set LIVE_SCRAPE=1 to run it.
import os
from pathlib import Path

LIVE_SCRAPE = os.environ.get("LIVE_SCRAPE") == "1"

if LIVE_SCRAPE:
    df = scrape_tokopedia("macbook", max_scroll=5, wait_time=3, use_path=False)
else:
    df = pd.read_csv(Path.cwd().parent / "data" / "tokopedia_raw.csv")
    print(f"Loaded {len(df)} saved records (set LIVE_SCRAPE=1 to scrape fresh)")


Loaded 60 saved records (set LIVE_SCRAPE=1 to scrape fresh)


In [5]:
#Hasil produk
df.head()

,Produk,Harga,Promo,Rating,Terjual,Penjual,Lokasi
0,CASE NEW MACBOOK PRO 16 inch M2 MAX 2023 COVER...,Rp115.000,NaN,NaN,NaN,luxer,Jakarta Barat
1,MOFT 2-in-1 Tas Laptop Sleeve & Stand |for Mac...,Rp2.000.000,NaN,4.9,100+ terjual,naladda_store,Jakarta Barat
2,Sodimm ddr3 pc12800/1600mhz 2gb hynix ori for ...,Rp80.000,NaN,5.0,17 terjual,Hpk,Jakarta Barat
3,Packing Kardus Khusus Pelanggan Rumah Apple - ...,Rp44.799.000,NaN,NaN,NaN,New RumahApple,Jakarta Pusat
4,Incase Reform Backpack For Macbook Up To 16 In...,Rp2.499.000,NaN,5.0,4 terjual,Incase Indonesia,Jakarta Pusat


In [6]:
# Save only when freshly scraped, so a demo run never overwrites the archived data.
if LIVE_SCRAPE:
    df.to_csv(Path.cwd().parent / "data" / "tokopedia_raw.csv", index=False, encoding="utf-8-sig")
    print("Saved to data/tokopedia_raw.csv")
else:
    print("Not scraped this run - archived CSV left untouched")


Not scraped this run - archived CSV left untouched


In [7]:
#https://drive.google.com/file/d/15K_dQ-dzHyePqUqqT6rxiNOiG0_jTs0o/view?usp=sharing